# Задачи 

## Планиране на градско строителство
Да приемем, че София е изправена пред бюджетен дефицит и общинският съвет обмисля как да увеличи приходите от данък сгради чрез преустройство на общински имоти. Проектът се състои от две части – разчистване на занемарени и повредени постройки и построяване на нови жилища.

1. В момента на планиране общината притежава 300 занемарени постройки, които могат да бъдат съборени. Всяка от тях освобождава 1000 кв. метра и събарянето й струва 3 000 EUR на постройка. 15 процента от освободеното пространство е предвидено за улици, тротоари и свободни пространства.
2. На освободените парцели общината може да построи четири вида нови жилищни сгради: еднофамилни (300 кв. метра), двуфамилни (500 кв. метра), три-фамилни (700 кв. метра) и четири-фамилни къщи (900 кв. метра). Очакваните данъчни постъпления са съответно 1 000 EUR, 1 700 EUR, 2 400 EUR и 2 800 EUR на година.
3. Поне 20 процента от новите постройки трябва да са еднофамилни къщи, двуфамилните домове трябва да съставляват поне 20 процента, а три и четири-фамилните домове трябва (заедно) да са поне една четвърт от всички нови постройки.
4. Строителните разходи за новите домове са съответно 50 000 EUR, 70 000 EUR, 130 000 EUR и 160 000 EUR.
5. Общината предвижда на финансира проекта чрез банков заем, който не може да надвишава 15 милиона EUR.

Колко от всеки вид къщи трябва да планира да построи общината, така че да постигне възможно най-високи данъчни постъпления?

Използвайте Excel за да намерите оптималното решение.

## Управление на ресторант

Управител на ресторант установява, че през изминалия сезон е реализирана значително по-малка от очакваната печалба. Една от основните причини е неефективно управление на наетия персонал.

При планиране на дейността за следващия период управителят се обръща към вас за съвет как да подобри дейността. Той ви предоставя следната информация:

1. Ресторантът наема сервитьори с различно ниво на опит, като заплащането на сервитьори с опит над 1 година е 15 EUR/час, съответно за опис от 6 месеца до 1 година е 10 EUR/час, а за опит под 6 месеца е 8 EUR/час.
2. Салонът е организиран в три секции – в първата секция са разположени маси за малки компании (до 4 човека), във втората секция – маси за средни по големина компании (до 8 човека), а в третата секция масите са за големи компании (над 8 човека).

След анализ на данни от предходни сезони вие установявате, че на час средно са били обслужвани по 40 малки, 30 средни и 25 големи маси. През следващия сезон не се очакват промени. Според оценката на двамата помощник-мениджъри, сервитьор с опит над 1 година успява да обслужи една малка маса за сумарно време от 5 минути, една средна маса – за 6 минути, а една голяма маса – за 7.5 минути. Аналогичните данни за сервитьор с опит между 6 месеца и 1 година са 7.5, 10 и 15 минути. Сервитьори с опит по-малък от 6 месеца обслужват само малки и средни маси, като съответното време е 15 минути за малка маса и 30 минути за средна маса.

Вашата задача е да минимизирате разходите за възнаграждения, изплатени на сервитьорите, при условие, че ресторантът винаги трябва да разполага с достатъчно на брой служители, които да обслужват заетите маси.

Намерете решението с Excel. Сверете решението си.

In [1]:
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

gp.setParam("LogToConsole", 0)

dt = pd.DataFrame(
    {
        "Tables": ["Small", "Medium", "Large"],
        "W1": [15, 20, 9999],
        "W2": [7.5, 10, 15],
        "W3": [5, 6, 7.5],
        "Required": [40, 30, 25]
    }
).set_index("Tables")

costs = [8, 10, 15]

dt

,W1,W2,W3,Required
Tables,,,,
Small,15,7.5,5.0,40
Medium,20,10.0,6.0,30
Large,9999,15.0,7.5,25


In [2]:
m = gp.Model("Restaurant")

w_exp =  ["W1", "W2", "W3"]

x = m.addVars(dt.index, w_exp, name="x")

# Objective

m.setObjective(
    gp.quicksum(costs[j] * x[i, w_exp[j]] for i in dt.index for j in range(len(w_exp))),
    GRB.MINIMIZE,
)

# Constraints

m.addConstrs(
    gp.quicksum((60 / dt.loc[i, w_exp[j]]) * x[i, w_exp[j]] for j in range(len(w_exp))) >= dt.loc[i, "Required"] for i in dt.index
)


m.write("restaurant.lp")

# Print the model

# with open("restaurant.lp", "r") as f:
#     print(f.read())

m.optimize()

# The solution as a DataFrame

solution = pd.DataFrame(
    {
        "Tables": [i for i in dt.index for j in range(len(w_exp))],
        "Workers": [w_exp[j] for i in dt.index for j in range(len(w_exp))],
        "Hours": [x[i, w_exp[j]].X for i in dt.index for j in range(len(w_exp))],
    }
).set_index("Tables")

solution.pivot_table(index="Tables", columns="Workers", values="Hours").fillna(0)

Workers,W1,W2,W3
Tables,,,
Large,0.0,0.0,3.125000
Medium,0.0,0.0,3.000000
Small,0.0,0.0,3.333333


## Оптимизация на кредитна политика на банка

Банка е в процес на определяне на своята кредитна политика за пет различни сектора. 

| Вид кредит | Лихва  | Дял на несъбираеми кредити |
| :--- | :---: | :---: |
| Потребителски| 15% | 10%  |
| Автомобили | 13% | 7%  |
| Недвижими имоти | 12%  | 3%  |
| Земеделие | 12.5%  | 5%  |
| Търговия | 10%  | 2%  |

Банката има бюджет от общо 12 милиона лева, които може да използва за кредитиране си при определяне на политиката трябва да се съобрази със следните регулации:

1. Поне 40% от заемите трябва да са за земеделие или търговия.
2. Заеми за недвижими имоти трябва да са поне половината от общата сума на потребителски кредити, кредити свързани с автомобили и недвижими имоти.
3. Според регулаторни изисквания дялът на несъбираемите кредити не трябва да надвишава 4% от всички кредити. Несъбираемите кредити не носят лихви.

Какъв е оптималният (най-висока печалба) план за банката?

Използвайте Excel за да намерите оптималното решение.